<a href="https://colab.research.google.com/github/Shehabmohammed598/ai-hw/blob/main/AI-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# تثبيت المكتبة
!pip install --upgrade gradio fuzzywuzzy

# استيراد المكتبات
import gradio as gr
from fuzzywuzzy import process

# قاعدة بيانات الأمراض الموسعة
disease_data = {
    "الملاريا": {
        "symptoms": ["حمى", "قشعريرة", "تعرق", "صداع"],
        "info": "الملاريا مرض طفيلي ينتقل عن طريق لدغات البعوض المصاب.",
        "treatment": "دواء الكلوروكين أو الأدوية المضادة للملاريا مثل الأرتيميسينين.",
        "instructions": "يُؤخذ الدواء وفق جرعات يحددها الطبيب بناءً على شدة الحالة."
    },
    "الإنفلونزا": {
        "symptoms": ["حمى", "سعال", "احتقان", "ألم عضلي"],
        "info": "الإنفلونزا عدوى فيروسية تصيب الجهاز التنفسي العلوي.",
        "treatment": "مسكنات الألم مثل الباراسيتامول، الراحة، والإكثار من السوائل.",
        "instructions": "يُؤخذ الباراسيتامول كل 4-6 ساعات مع الالتزام بالجرعة اليومية الموصى بها."
    },
    "كوفيد-19": {
        "symptoms": ["حمى", "سعال", "ضيق تنفس", "إرهاق", "فقدان حاسة الشم", "فقدان حاسة التذوق"],
        "info": "كوفيد-19 مرض فيروسي يسببه فيروس كورونا المستجد.",
        "treatment": "علاج الأعراض مثل خافضات الحرارة، وفي الحالات الشديدة قد يُستخدم الأكسجين أو الأدوية المضادة للفيروسات.",
        "instructions": "استشر الطبيب لتحديد العلاج بناءً على شدة الأعراض."
    },
    "التسمم الغذائي": {
        "symptoms": ["غثيان", "قيء", "إسهال", "تشنجات في البطن"],
        "info": "التسمم الغذائي يحدث بسبب تناول طعام ملوث بالبكتيريا أو الفيروسات.",
        "treatment": "تعويض السوائل المفقودة، ومضادات التشنج إذا لزم الأمر.",
        "instructions": "تناول أملاح الإماهة الفموية حسب الإرشادات، واستشر الطبيب إذا استمرت الأعراض."
    },
    "التهاب الأذن الوسطى": {
        "symptoms": ["ألم في الأذن", "فقدان السمع", "حمى", "إفرازات من الأذن"],
        "info": "التهاب الأذن الوسطى عدوى تصيب الأذن الوسطى وعادةً تحدث بعد نزلات البرد.",
        "treatment": "مضادات حيوية مثل الأموكسيسيلين إذا كان السبب بكتيريًا.",
        "instructions": "يُؤخذ الدواء حسب الجرعة المحددة من قبل الطبيب لمدة 7-10 أيام."
    },
    "حصى الكلى": {
        "symptoms": ["ألم شديد في الجنب", "دم في البول", "غثيان", "قيء"],
        "info": "حصى الكلى هي تراكم معادن صلبة تتكون في الكلى.",
        "treatment": "شرب الكثير من السوائل، أو أدوية لتسكين الألم، أو أحيانًا تفتيت الحصى بالموجات الصوتية.",
        "instructions": "استشر الطبيب لتحديد العلاج المناسب، خاصة إذا كانت الحصى كبيرة الحجم."
    }
}

# قائمة بالكلمات الصحيحة (تصحيح الأخطاء الإملائية)
valid_symptoms = set(word for disease in disease_data.values() for word in disease["symptoms"])

# دالة لتصحيح الأخطاء الإملائية في الأعراض
def correct_symptoms(input_symptoms):
    corrected = []
    for symptom in input_symptoms:
        # استخدام مكتبة fuzzywuzzy لتصحيح الأعراض
        best_match, score = process.extractOne(symptom, valid_symptoms)
        if score > 80:  # إذا كانت نسبة التشابه عالية، نعتبرها صحيحة
            corrected.append(best_match)
    return corrected

# وظيفة لتحديد المرض بناءً على الأعراض
def predict_disease(symptom_list):
    if not symptom_list:
        return "الرجاء إدخال الأعراض للتشخيص."

    # تصحيح الأعراض
    corrected_symptoms = correct_symptoms(symptom_list)

    if not corrected_symptoms:
        return "لم يتم التعرف على الأعراض المدخلة. الرجاء إدخال أعراض أكثر وضوحًا."

    for disease, details in disease_data.items():
        # تحقق إذا كانت كل الأعراض المدخلة (بعد التصحيح) موجودة ضمن أعراض المرض
        if all(symptom in details["symptoms"] for symptom in corrected_symptoms):
            return (f"التشخيص: {disease}\n"
                    f"وصف: {details['info']}\n"
                    f"العلاج: {details['treatment']}\n"
                    f"كيفية أخذ العلاج: {details['instructions']}")

    return "لم يتم العثور على مرض تنطبق عليه جميع الأعراض المدخلة."

# واجهة المستخدم
with gr.Blocks() as interface:
    gr.Markdown("## نظام تشخيص الأمراض")
    gr.Markdown("### أدخل الأعراض للحصول على التشخيص.")

    # حقل ديناميكي لإضافة الأعراض
    symptom_inputs = gr.State([])  # قائمة الأعراض
    symptoms_output = gr.Textbox(label="الأعراض المُدخلة", lines=3, interactive=False)

    # وظيفة لإضافة الأعراض
    def add_symptom(symptom, symptom_list):
        if symptom.strip():  # التحقق من أن العرض غير فارغ
            symptom_list.append(symptom.strip())
        return symptom_list, ", ".join(symptom_list), ""

    # وظيفة لإعادة تعيين الأعراض
    def reset_symptoms():
        return [], "", ""

    # إدخال عرض واحد في كل مرة
    symptom_input = gr.Textbox(label="أدخل عرضًا (مثل: حمى، سعال)")
    add_button = gr.Button("إضافة عرض")
    reset_button = gr.Button("إعادة تعيين الأعراض")
    add_button.click(add_symptom, inputs=[symptom_input, symptom_inputs], outputs=[symptom_inputs, symptoms_output, symptom_input])
    reset_button.click(reset_symptoms, outputs=[symptom_inputs, symptoms_output, symptom_input])

    # زر للتشخيص
    diagnose_button = gr.Button("تشخيص")
    diagnosis_output = gr.Textbox(label="التشخيص", lines=6, interactive=False)
    diagnose_button.click(predict_disease, inputs=[symptom_inputs], outputs=[diagnosis_output])

# تشغيل التطبيق
interface.launch(share=True)

/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdeac5cebc571dfc6c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
